<a href="https://colab.research.google.com/github/Harnoor001/UML501-TIET-MachineLearning/blob/main/UML501_Assignment_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================================
# UML501 - ASSIGNMENT 3 (FINAL CLEAN CODE)
# =========================================

import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression

# =========================================
# Q1: 5-FOLD CV (LEAST SQUARE FROM SCRATCH)
# =========================================

print("\n===== Q1: K-Fold Cross Validation =====")

# Upload dataset manually in Colab
df = pd.read_csv("USA_Housing.csv")

X = df.drop("Price", axis=1).values
y = df["Price"].values

# Scaling
scaler = StandardScaler()
X = scaler.fit_transform(X)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

best_r2 = -np.inf
best_beta = None

for i, (train_idx, test_idx) in enumerate(kf.split(X)):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Add bias
    X_train_b = np.c_[np.ones(X_train.shape[0]), X_train]
    X_test_b = np.c_[np.ones(X_test.shape[0]), X_test]

    # Least squares solution
    beta = np.linalg.inv(X_train_b.T @ X_train_b) @ X_train_b.T @ y_train

    y_pred = X_test_b @ beta
    r2 = r2_score(y_test, y_pred)

    print(f"Fold {i+1} R2 Score: {r2:.4f}")

    if r2 > best_r2:
        best_r2 = r2
        best_beta = beta

print("Best R2:", best_r2)

# Train on 70% and test on 30% using best beta
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

X_train_b = np.c_[np.ones(X_train.shape[0]), X_train]
X_test_b = np.c_[np.ones(X_test.shape[0]), X_test]

y_pred = X_test_b @ best_beta
print("Final Test R2:", r2_score(y_test, y_pred))


# =========================================
# Q2: VALIDATION SET (GRADIENT DESCENT)
# =========================================

print("\n===== Q2: Gradient Descent with Validation =====")

# Split: 56% train, 14% val, 30% test
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42)

def gradient_descent(X, y, lr, epochs=1000):
    n, m = X.shape
    w = np.zeros(m)

    for _ in range(epochs):
        y_pred = X @ w
        grad = (2/n) * (X.T @ (y_pred - y))
        w -= lr * grad

    return w

learning_rates = [0.001, 0.01, 0.1, 1]

best_r2 = -np.inf
best_w = None

for lr in learning_rates:
    w = gradient_descent(X_train, y_train, lr)

    val_pred = X_val @ w
    test_pred = X_test @ w

    val_r2 = r2_score(y_val, val_pred)
    test_r2 = r2_score(y_test, test_pred)

    print(f"LR: {lr}, Val R2: {val_r2:.4f}, Test R2: {test_r2:.4f}")

    if val_r2 > best_r2:
        best_r2 = val_r2
        best_w = w

print("Best Validation R2:", best_r2)


# =========================================
# Q3: CAR DATASET PREPROCESSING
# =========================================

print("\n===== Q3: Car Dataset =====")

cols = ["symboling","normalized_losses","make","fuel_type","aspiration","num_doors",
        "body_style","drive_wheels","engine_location","wheel_base","length","width",
        "height","curb_weight","engine_type","num_cylinders","engine_size",
        "fuel_system","bore","stroke","compression_ratio","horsepower","peak_rpm",
        "city_mpg","highway_mpg","price"]

df = pd.read_csv("imports-85.data", names=cols)

# Replace ? with NaN
df.replace("?", np.nan, inplace=True)

# Convert numeric columns
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='ignore')

# Drop rows where price is NaN
df = df.dropna(subset=["price"])

# Fill NaN with mean
df = df.fillna(df.mean(numeric_only=True))

# num_doors & num_cylinders mapping
mapping = {"two":2, "three":3, "four":4, "five":5, "six":6, "eight":8, "twelve":12}
df["num_doors"] = df["num_doors"].map(mapping)
df["num_cylinders"] = df["num_cylinders"].map(mapping)

# fuel_system: pfi → 1 else 0
df["fuel_system"] = df["fuel_system"].apply(lambda x: 1 if "pfi" in str(x) else 0)

# engine_type: ohc → 1 else 0
df["engine_type"] = df["engine_type"].apply(lambda x: 1 if "ohc" in str(x) else 0)

# Dummy encoding
df = pd.get_dummies(df, columns=["body_style", "drive_wheels"], drop_first=True)

# Label encoding
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
for col in ["make", "aspiration", "engine_location", "fuel_type"]:
    df[col] = le.fit_transform(df[col].astype(str))

# Split
X = df.drop("price", axis=1)
y = df["price"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)

# Linear Regression
model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Linear Regression R2:", r2_score(y_test, y_pred))

# PCA
pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X_scaled)

X_train, X_test, y_train, y_test = train_test_split(X_pca, y, test_size=0.3, random_state=42)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("After PCA R2:", r2_score(y_test, y_pred))

# =========================================
# END
# =========================================